In [ ]:
!pip install git+https://github.com/facebookresearch/segment-anything.git \
                opencv-python pillow matplotlib numpy tqdm scikit-image

In [ ]:
!pip -q install timm

In [ ]:
# ──────────────────────────────────────────────────────────────
# 1. PATHS –  CHANGE TO MATCH YOUR DISK LAYOUT
# ──────────────────────────────────────────────────────────────
from pathlib import Path

COCO_DIR    = Path("")      # only the JPGs are needed
SPLIT       = "train2017"
IMG_ID      = 109                    # 000000000109.jpg
TARGET_SIZE = 224                    # 0 = skip resize
CHECKPOINT  = Path("sam_vit_b_01ec64.pth")   # download from Meta and place here 
MODEL_TYPE  = "vit_b"                # vit_h | vit_l | vit_b

IMG_DIR     = COCO_DIR / SPLIT

In [ ]:
# ──────────────────────────────────────────────────────────────
# 2. LOAD THE PHOTO
# ──────────────────────────────────────────────────────────────
import numpy as np, matplotlib.pyplot as plt, cv2
from PIL import Image

file_name = f"{IMG_ID:012d}.jpg"
img_path  = "WhatsApp.jpeg"
#IMG_DIR / file_name
orig_img  = cv2.imread(str(img_path))           # BGR
orig_rgb  = cv2.cvtColor(orig_img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(6,6))
plt.imshow(orig_rgb); plt.axis("off"); plt.title("Original photo")
plt.show()

In [ ]:
# ──────────────────────────────────────────────────────────────
# 3. RUN SAM
# ──────────────────────────────────────────────────────────────
import torch
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

# 1) pick device
device = "cpu"

# 2) build model WITHOUT loading checkpoint inside the library
sam = sam_model_registry[MODEL_TYPE](checkpoint=None)

# 3) load checkpoint with map_location (critical)
ckpt = torch.load(CHECKPOINT, map_location="cpu")  # or map_location=device

# some checkpoints wrap the actual state dict
if isinstance(ckpt, dict) and "model" in ckpt:
    state_dict = ckpt["model"]
elif isinstance(ckpt, dict) and "state_dict" in ckpt:
    state_dict = ckpt["state_dict"]
else:
    state_dict = ckpt

sam.load_state_dict(state_dict, strict=True)
sam.to(device)

mask_generator = SamAutomaticMaskGenerator(
    sam,
    points_per_side=32,
    pred_iou_thresh=0.9,
    stability_score_thresh=0.92,
    min_mask_region_area=0
)

masks = mask_generator.generate(orig_rgb)
print(len(masks))

In [ ]:
# ──────────────────────────────────────────────────────────────
# 4. VISUALISE SOME MASKS
# ──────────────────────────────────────────────────────────────
N = min(30, len(masks))
cols, rows = 6, (N+5)//6
plt.figure(figsize=(cols*2, rows*2))

for i, m in enumerate(sorted(masks, key=lambda x: -x["area"])[:N]):
    plt.subplot(rows, cols, i+1)
    plt.imshow(m["segmentation"], cmap="gray")
    plt.title(f"{i}  area={m['area']}")
    plt.axis("off")

plt.tight_layout(); plt.show()

In [ ]:
import numpy as np

# ---- CALIBRATE THESE ONCE for your camera/layout ----
x_left_frac  = 0.16
x_right_frac = 0.81

y_top_frac = 0.105     # y-position (as fraction of H) near the liquid of the top row
y_bot_frac = 0.38     # y-position near the liquid of the bottom row

# Box size relative to spacing / image size
box_w_scale = 0.55    # box width = box_w_scale * tube spacing (dx)
box_up_frac = 0.04    # how far box extends above row y (fraction of H)
box_dn_frac = 0.1    # how far box extends below row y (fraction of H)

In [ ]:
import matplotlib.pyplot as plt
import cv2
import numpy as np

def preview_rois(rgb):
    """
    Overlay the 16 hard-coded ROIs so you can verify their
    position and size.  No SAM is run here – it is only a drawing
    utility.
    """
    H, W = rgb.shape[:2]
    centers, boxes = tube_centers_and_boxes(H, W)

    # draw on a copy (OpenCV uses BGR, matplotlib uses RGB)
    vis = cv2.cvtColor(rgb.copy(), cv2.COLOR_RGB2BGR)

    # rectangles
    for (x0, y0, x1, y1) in boxes:
        cv2.rectangle(
            vis,
            (int(x0), int(y0)),
            (int(x1), int(y1)),
            color=(0, 255, 0),  # green
            thickness=2,
        )

    # center prompts
    for (cx, cy) in centers:
        cv2.circle(
            vis,
            (int(cx), int(cy)),
            radius=5,
            color=(0, 0, 255),  # red
            thickness=-1,
        )

    # show
    plt.figure(figsize=(8, 6))
    plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.title("Tube ROIs (green) + SAM prompt points (red)")
    plt.show()

In [ ]:
def tube_centers_and_boxes(H, W):
    x0, x1 = x_left_frac * W, x_right_frac * W
    xs = np.linspace(x0, x1, 8)

    y_top = y_top_frac * H
    y_bot = y_bot_frac * H

    dx = (x1 - x0) / 7.0
    bw = box_w_scale * dx
    up = box_up_frac * H
    dn = box_dn_frac * H

    centers = []
    boxes = []
    for y in [y_top, y_bot]:
        for x in xs:
            x0b, y0b = x - bw/2, y - up
            x1b, y1b = x + bw/2, y + dn

            x0b = int(np.clip(x0b, 0, W-1)); x1b = int(np.clip(x1b, 0, W-1))
            y0b = int(np.clip(y0b, 0, H-1)); y1b = int(np.clip(y1b, 0, H-1))
            if x1b <= x0b: x1b = min(W-1, x0b+1)
            if y1b <= y0b: y1b = min(H-1, y0b+1)

            centers.append((x, y))
            boxes.append(np.array([x0b, y0b, x1b, y1b], dtype=np.float32))
    return np.array(centers, dtype=np.float32), boxes

In [ ]:
bgr = cv2.imread("WhatsApp.jpeg")          # or any test frame
rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

preview_rois(rgb)      # <-- run this once after you tweaked the 4 fractions

In [ ]:
import cv2
import torch
from segment_anything import sam_model_registry, SamPredictor

img_path = "WhatsApp.jpeg"
bgr = cv2.imread(img_path)
rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
H, W = rgb.shape[:2]

# 1) pick device
device = "cpu"

# 2) build model WITHOUT loading checkpoint inside the library
sam = sam_model_registry[MODEL_TYPE](checkpoint=None)

# 3) load checkpoint with map_location (critical)
ckpt = torch.load(CHECKPOINT, map_location="cpu")  # or map_location=device

# some checkpoints wrap the actual state dict
if isinstance(ckpt, dict) and "model" in ckpt:
    state_dict = ckpt["model"]
elif isinstance(ckpt, dict) and "state_dict" in ckpt:
    state_dict = ckpt["state_dict"]
else:
    state_dict = ckpt

sam.load_state_dict(state_dict, strict=True)
sam.to(device)

predictor = SamPredictor(sam)
predictor.set_image(rgb)

centers, boxes = tube_centers_and_boxes(H, W)
len(boxes)

In [ ]:
import cv2
import numpy as np

def largest_component(mask_bool):
    m = (mask_bool.astype(np.uint8) * 255)
    num, labels = cv2.connectedComponents(m, connectivity=8)
    if num <= 1:
        return mask_bool
    areas = [(labels == i).sum() for i in range(1, num)]
    best = 1 + int(np.argmax(areas))
    return (labels == best)

def inner_core_from_tube_mask(tube_mask, keep_core_frac=0.45):
    """
    Get a 'core' region inside the tube (removes walls/printing).
    keep_core_frac=0.45 keeps pixels with distance >= 45% of max distance.
    """
    tube_u8 = tube_mask.astype(np.uint8)
    dist = cv2.distanceTransform(tube_u8, distanceType=cv2.DIST_L2, maskSize=5)
    m = dist.max()
    if m <= 1e-6:
        return tube_mask.copy()
    core = tube_mask & (dist >= keep_core_frac * m)
    return core

def chemical_by_color(crop_rgb, tube_mask_crop):
    hsv = cv2.cvtColor(crop_rgb, cv2.COLOR_RGB2HSV)
    S = hsv[..., 1]
    V = hsv[..., 2]

    s_vals = S[tube_mask_crop]
    if s_vals.size == 0:
        return np.zeros_like(tube_mask_crop, dtype=bool)

    s_thr = max(25, int(np.percentile(s_vals, 70)))
    v_thr = 245
    chem = tube_mask_crop & (S >= s_thr) & (V <= v_thr)

    chem = cv2.morphologyEx(chem.astype(np.uint8), cv2.MORPH_OPEN,
                            cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3,3)),
                            iterations=1).astype(bool)
    chem = cv2.morphologyEx(chem.astype(np.uint8), cv2.MORPH_CLOSE,
                            cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5)),
                            iterations=2).astype(bool)
    chem = largest_component(chem)
    return chem



In [ ]:
import cv2
import numpy as np

def keep_lowest_component(mask_bool):
    m = mask_bool.astype(np.uint8)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
    if n <= 1:
        return mask_bool

    # choose component with the largest bottom y (top+height-1)
    best = 1
    best_ymax = -1
    for i in range(1, n):
        top = stats[i, cv2.CC_STAT_TOP]
        h   = stats[i, cv2.CC_STAT_HEIGHT]
        ymax = top + h - 1
        if ymax > best_ymax:
            best_ymax = ymax
            best = i
    return labels == best

In [ ]:
def inner_core_from_tube_mask(tube_mask, keep_core_frac=0.55):
    tube_u8 = tube_mask.astype(np.uint8)
    dist = cv2.distanceTransform(tube_u8, distanceType=cv2.DIST_L2, maskSize=5)
    m = dist.max()
    if m <= 1e-6:
        return tube_mask.copy()
    return tube_mask & (dist >= keep_core_frac * m)

In [ ]:
def boundary_by_air_difference(crop_rgb, core_mask):
    H, W = core_mask.shape
    lab = cv2.cvtColor(crop_rgb, cv2.COLOR_RGB2LAB).astype(np.float32)

    # "air reference" from top band inside core
    y0 = int(0.05 * H)
    y1 = int(0.25 * H)
    top_band = np.zeros_like(core_mask)
    top_band[y0:y1] = True
    ref_mask = core_mask & top_band

    if ref_mask.sum() < 50:
        return int(0.65 * H)  # fallback

    ref = lab[ref_mask].mean(axis=0)  # [L,a,b]
    diff = np.linalg.norm(lab - ref, axis=2)  # per-pixel difference to "air"

    # row profile
    prof = np.full(H, np.nan, dtype=np.float32)
    for y in range(H):
        row = core_mask[y]
        if row.any():
            prof[y] = diff[y, row].mean()

    idx = np.where(~np.isnan(prof))[0]
    if idx.size < 10:
        return int(0.65 * H)

    # fill NaNs + smooth
    prof_f = prof.copy()
    prof_f[np.isnan(prof_f)] = np.interp(np.where(np.isnan(prof_f))[0], idx, prof_f[idx])
    prof_s = cv2.GaussianBlur(prof_f.reshape(-1,1), (1, 21), 0).ravel()

    # pick boundary where gradient is strongest (ignore very top/bottom)
    g = np.abs(np.gradient(prof_s))
    y_min = int(0.10 * H)
    y_max = int(0.95 * H)
    y_star = y_min + int(np.argmax(g[y_min:y_max]))

    # if gradient is extremely weak, use a conservative default
    if g[y_star] < 0.2:
        y_star = int(0.65 * H)

    return y_star

In [ ]:
def chemical_from_crop_strict(
    crop_rgb,
    tube_mask_crop,
    min_area_frac=0.06,        # guarantee at least 6% of tube area (adjust)
    bottomness_frac=0.50,      # cue must reach lower 40% of crop to be trusted
    keep_core_frac=0.55
):
    H, W = tube_mask_crop.shape
    tube_area = int(tube_mask_crop.sum())
    if tube_area == 0:
        return np.zeros_like(tube_mask_crop, dtype=bool)

    core = inner_core_from_tube_mask(tube_mask_crop, keep_core_frac=keep_core_frac)

    # --- Step 1: try a color cue on the core (your existing method) ---
    chem_cue = chemical_by_color(crop_rgb, core)          # uses HSV, but core avoids walls/printing
    chem_cue = keep_lowest_component(chem_cue)            # keep only the lowest blob

    cue_area = int(chem_cue.sum())
    cue_ymax = np.where(chem_cue)[0].max() if cue_area > 0 else -1
    cue_ok = (cue_area > 0) and (cue_ymax >= int(bottomness_frac * H))

    if cue_ok:
        # Convert cue into a full liquid region by using its TOP as boundary and filling below
        y_star = int(np.where(chem_cue)[0].min())
    else:
        # --- Step 2: boundary from air-difference profile (works for clear liquid) ---
        y_star = boundary_by_air_difference(crop_rgb, core)

    yy = np.arange(H)[:, None]
    chem = tube_mask_crop & (yy >= y_star)

    # --- Step 3: ensure not tiny (hard guarantee) ---
    min_area = int(min_area_frac * tube_area)
    if chem.sum() < min_area:
        # take bottom portion of the tube big enough to satisfy min_area
        ys = np.where(tube_mask_crop)[0]
        y_sorted = np.sort(ys)
        # pick y so that we keep at least min_area tube pixels at the bottom
        # approximate by choosing a y quantile
        q = max(0.0, 1.0 - (min_area / tube_area))
        y_q = int(np.quantile(y_sorted, q))
        chem = tube_mask_crop & (yy >= y_q)

    # Cleanup: keep lowest connected component (liquid should be bottom-most)
    chem = keep_lowest_component(chem)
    return chem

In [ ]:

def split_parts(crop_rgb, tube_mask_crop, chem_mask_crop, dilate_px=5):
    """
    Returns:
      background_mask, tube_overlap_mask, tube_nonoverlap_mask, chem_mask
    where overlap/nonoverlap are tube *wall/marking* pixels (tube minus chem),
    split by whether they are near the chemical region (dilated chemical).
    """
    tube_wall = tube_mask_crop & (~chem_mask_crop)

    k = max(3, int(dilate_px) | 1)  # odd
    ker = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
    chem_dil = cv2.dilate(chem_mask_crop.astype(np.uint8), ker, iterations=1).astype(bool)

    tube_overlap = tube_wall & chem_dil
    tube_nonoverlap = tube_wall & (~chem_dil)
    background = ~tube_mask_crop

    return background, tube_overlap, tube_nonoverlap, chem_mask_crop

In [ ]:
import cv2
import numpy as np

def boundary_by_air_diff_bottomup(crop_rgb, core_mask, consec=12, k_sigma=2.0):
    """
    Returns y_star (int) fill boundary such that liquid is y >= y_star.
    If no stable region found, returns None.
    """
    H, W = core_mask.shape
    lab = cv2.cvtColor(crop_rgb, cv2.COLOR_RGB2LAB).astype(np.float32)

    # Air reference from top band inside core
    y0, y1 = int(0.05 * H), int(0.25 * H)
    ref_mask = core_mask.copy()
    ref_mask[:y0] = False
    ref_mask[y1:] = False

    if ref_mask.sum() < 50:
        return None

    ref = lab[ref_mask].mean(axis=0)  # [L,a,b]
    diff = np.linalg.norm(lab - ref, axis=2)

    # Row profile: use MEDIAN (more robust than mean)
    prof = np.full(H, np.nan, dtype=np.float32)
    for y in range(H):
        row = core_mask[y]
        if row.any():
            prof[y] = np.median(diff[y, row])

    idx = np.where(~np.isnan(prof))[0]
    if idx.size < 10:
        return None

    # Fill + smooth
    prof_f = prof.copy()
    prof_f[np.isnan(prof_f)] = np.interp(np.where(np.isnan(prof_f))[0], idx, prof_f[idx])
    prof_s = cv2.GaussianBlur(prof_f.reshape(-1,1), (1, 21), 0).ravel()

    # Threshold derived from top-band baseline (air-like)
    base = prof_s[y0:y1]
    mu, sig = float(np.mean(base)), float(np.std(base) + 1e-6)
    thr = mu + k_sigma * sig

    # Bottom-up persistence: find first run of >= consec rows above thr
    run = 0
    y_start = None
    for y in range(H-1, -1, -1):
        if prof_s[y] > thr:
            run += 1
            if run >= consec:
                y_start = y  # boundary at the top of this stable region
        else:
            run = 0
            if y_start is not None:
                break

    return y_start

In [ ]:
import os
from PIL import Image

os.makedirs("crops", exist_ok=True)
os.makedirs("masks", exist_ok=True)

all_results = []

for i, (center, box) in enumerate(zip(centers, boxes)):
    x0, y0, x1, y1 = box.astype(int)
    crop = rgb[y0:y1, x0:x1].copy()

    # --- SAM tube mask on full image, then crop it ---
    cx, cy = center
    point_coords = np.array([[cx, cy]], dtype=np.float32)
    point_labels = np.array([1], dtype=np.int32)

    masks, scores, _ = predictor.predict(
        point_coords=point_coords,
        point_labels=point_labels,
        box=box,
        multimask_output=True
    )
    kbest = int(np.argmax(scores))
    tube_mask_full = masks[kbest].astype(bool)
    tube_mask_crop = tube_mask_full[y0:y1, x0:x1]

    # (optional) keep largest component inside crop to avoid stray pixels
    tube_mask_crop = largest_component(tube_mask_crop)

    # --- chemical mask inside the tube mask ---
    chem_mask = chemical_from_crop_strict(crop, tube_mask_crop)

    # --- split to your 3 parts (+ chemical) ---
    # dilate_px should scale with crop size; this is a decent default:
    dilate_px = max(3, int(0.03 * (x1-x0)))
    bg, tube_ov, tube_non, chem = split_parts(crop, tube_mask_crop, chem_mask, dilate_px=dilate_px)

    # --- label map: 0 bg, 1 tube_non, 2 tube_overlap, 3 chem ---
    label = np.zeros(bg.shape, dtype=np.uint8)
    label[tube_non] = 1
    label[tube_ov] = 2
    label[chem] = 3

    # save
    Image.fromarray(crop).save(f"crops/tube_{i:02d}.png")
    Image.fromarray((tube_mask_crop.astype(np.uint8)*255)).save(f"masks/tube_{i:02d}_tube.png")
    Image.fromarray((chem.astype(np.uint8)*255)).save(f"masks/tube_{i:02d}_chem.png")
    Image.fromarray((tube_ov.astype(np.uint8)*255)).save(f"masks/tube_{i:02d}_tube_overlap.png")
    Image.fromarray((tube_non.astype(np.uint8)*255)).save(f"masks/tube_{i:02d}_tube_nonoverlap.png")
    Image.fromarray(label).save(f"masks/tube_{i:02d}_label.png")

    all_results.append((crop, label, float(scores[kbest])))

print("Done. Saved to ./crops and ./masks")

In [ ]:
import matplotlib.pyplot as plt

# show a few tubes
for idx in [0, 1,2,3,4,5,6, 7, 8,9,10,11,12,13,14, 15]:
    crop, label, sc = all_results[idx]

    plt.figure(figsize=(10,3))
    plt.subplot(1,3,1); plt.imshow(crop); plt.axis("off"); plt.title(f"tube {idx} crop")
    plt.subplot(1,3,2); plt.imshow(label, vmin=0, vmax=3); plt.axis("off"); plt.title("label (0 bg,1 tube,2 ov,3 chem)")
    plt.subplot(1,3,3)
    overlay = crop.copy()
    overlay[label==3] = (overlay[label==3]*0.4 + np.array([255,0,0])*0.6).astype(np.uint8)   # chem red
    overlay[label==2] = (overlay[label==2]*0.4 + np.array([0,255,0])*0.6).astype(np.uint8)   # overlap green
    overlay[label==1] = (overlay[label==1]*0.6 + np.array([0,0,255])*0.4).astype(np.uint8)   # tube blue
    plt.imshow(overlay); plt.axis("off"); plt.title(f"overlay (SAM score {sc:.2f})")
    plt.show()